# 02 Pipeline (Full Read Pipeline Template)

This is the **full-read pipeline template**. Clone Read and Write blocks as needed for many-to-many pipelines: every governed source is read in full, while each target can still use its governed write strategy such as overwrite, append, SCD1, or SCD2.


## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.2.0 | Voyce | 6 Aug 2026 |

This redesigned version has local structural and public-API compatibility validation only. Run it in your configured Fabric workspace before recording a new runtime test.


# 0. Environment

Run the shared Fabric configuration and import the public APIs used by this pipeline.


In [ ]:
%run 00_env_config


In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_schema,
    check_sensitive_data,
    check_source_drift,
    pipeline_read,
    pipeline_write,
    profile_table,
    resolve_table_id,
    widget_select_data_contract,
    widget_view_catalogue,
    write_lakehouse_table,
    write_warehouse_table,
)


# 1. Data Contract

Select the Data Contracts to test with this pipeline. Production automatically uses activated Data Contracts.


In [ ]:
CONTRACTS = widget_select_data_contract()


# 2. Full Read

Each cloneable Read block reads the complete governed source, then runs Freshness, Schema, Data Quality, and a full persisted profile. This template does not perform source-side incremental reads.


In [ ]:
# Keep each named source together so later Transform, Source Drift, and Lineage steps can reuse it.
sources = {}


## READ 1 — Orders


In [ ]:
READ_NAME = "orders"
READ_STORE = "source"
READ_SCHEMA = "demo"
READ_TABLE = "orders"
READ_QUERY = None

# Read the whole source table and return both the DataFrame and its FabricOps table_id.
source = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

df = source["dataframe"]
table_id = source["table_id"]

# Stop here if this source is older than the allowed Freshness rule.
freshness_result = check_freshness(table_id, raise_on_failure=True)

# Stop here if the source columns or data types no longer match the contract.
schema_result = check_schema(df, table_id=table_id, raise_on_failure=True)

# Run the row-level Data Quality rules.
# Keep both the checked DataFrame and the failed rows so you can inspect or save them if needed.
dq_result = check_dq(df, table_id=table_id, raise_on_failure=True)
dq_df = dq_result.get("dataframe", df)
dq_failed_values = dq_result.get("failed_values")

# Refresh the stored profile for the full source table.
profile_result = profile_table(table_id=table_id)

# Save this source by name. Later cells use it for Transform, Source Drift, and Lineage.
sources[READ_NAME] = source

# Optional development inspection — uncomment only when needed.
# display(df)                              # View the source data
# display(profile_result["profile"])       # View the latest full-table profile
# display(dq_df)                           # View the source with DQ status columns
# display(dq_failed_values)                # View the rows/values that failed DQ

# Optional failure persistence — save DQ failures somewhere your project owns.
# write_lakehouse_table(
#     dq_failed_values,
#     "dq_failed_values",
#     store="unified",
#     schema="support",
#     mode="append",
# )
# write_warehouse_table(
#     dq_failed_values,
#     schema="support",
#     table_name="dq_failed_values",
#     store="warehouse",
#     mode="append",
# )

# Optional metadata inspection — show what FabricOps knows about this table_id.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=table_id)


## READ 2 — Products


In [ ]:
READ_NAME = "products"
READ_STORE = "source"
READ_SCHEMA = "demo"
READ_TABLE = "products"
READ_QUERY = None

# Read the whole source table and return both the DataFrame and its FabricOps table_id.
source = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

df = source["dataframe"]
table_id = source["table_id"]

# Stop here if this source is older than the allowed Freshness rule.
freshness_result = check_freshness(table_id, raise_on_failure=True)

# Stop here if the source columns or data types no longer match the contract.
schema_result = check_schema(df, table_id=table_id, raise_on_failure=True)

# Run the row-level Data Quality rules.
# Keep both the checked DataFrame and the failed rows so you can inspect or save them if needed.
dq_result = check_dq(df, table_id=table_id, raise_on_failure=True)
dq_df = dq_result.get("dataframe", df)
dq_failed_values = dq_result.get("failed_values")

# Refresh the stored profile for the full source table.
profile_result = profile_table(table_id=table_id)

# Save this source by name. Later cells use it for Transform, Source Drift, and Lineage.
sources[READ_NAME] = source

# Optional development inspection — uncomment only when needed.
# display(df)                              # View the source data
# display(profile_result["profile"])       # View the latest full-table profile
# display(dq_df)                           # View the source with DQ status columns
# display(dq_failed_values)                # View the rows/values that failed DQ

# Optional failure persistence — save DQ failures somewhere your project owns.
# write_lakehouse_table(
#     dq_failed_values,
#     "dq_failed_values",
#     store="unified",
#     schema="support",
#     mode="append",
# )
# write_warehouse_table(
#     dq_failed_values,
#     schema="support",
#     table_name="dq_failed_values",
#     store="warehouse",
#     mode="append",
# )

# Optional metadata inspection — show what FabricOps knows about this table_id.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=table_id)


## READ 3 — Order History


In [ ]:
READ_NAME = "history"
READ_STORE = "product"
READ_SCHEMA = "demo"
READ_TABLE = "order_history"
READ_QUERY = None

# Read the whole source table and return both the DataFrame and its FabricOps table_id.
source = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

df = source["dataframe"]
table_id = source["table_id"]

# Stop here if this source is older than the allowed Freshness rule.
freshness_result = check_freshness(table_id, raise_on_failure=True)

# Stop here if the source columns or data types no longer match the contract.
schema_result = check_schema(df, table_id=table_id, raise_on_failure=True)

# Run the row-level Data Quality rules.
# Keep both the checked DataFrame and the failed rows so you can inspect or save them if needed.
dq_result = check_dq(df, table_id=table_id, raise_on_failure=True)
dq_df = dq_result.get("dataframe", df)
dq_failed_values = dq_result.get("failed_values")

# Refresh the stored profile for the full source table.
profile_result = profile_table(table_id=table_id)

# Save this source by name. Later cells use it for Transform, Source Drift, and Lineage.
sources[READ_NAME] = source

# Optional development inspection — uncomment only when needed.
# display(df)                              # View the source data
# display(profile_result["profile"])       # View the latest full-table profile
# display(dq_df)                           # View the source with DQ status columns
# display(dq_failed_values)                # View the rows/values that failed DQ

# Optional failure persistence — save DQ failures somewhere your project owns.
# write_lakehouse_table(
#     dq_failed_values,
#     "dq_failed_values",
#     store="unified",
#     schema="support",
#     mode="append",
# )
# write_warehouse_table(
#     dq_failed_values,
#     schema="support",
#     table_name="dq_failed_values",
#     store="warehouse",
#     mode="append",
# )

# Optional metadata inspection — show what FabricOps knows about this table_id.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=table_id)


# 3. Transform

Keep project transformations as ordinary, readable PySpark. Reuse the named source DataFrames to build one or more target DataFrames.


In [ ]:
# Pull the DataFrames back out by the names used in the Read blocks.
orders_df = sources["orders"]["dataframe"]
products_df = sources["products"]["dataframe"]
history_df = sources["history"]["dataframe"]

# Build the project-specific transformation using normal PySpark.
history_summary_df = (
    history_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("historical_order_count"),
        F.sum("net_amount").alias("historical_net_amount"),
        F.max("order_datetime").alias("latest_historical_order_datetime"),
    )
)

transformed_df = (
    orders_df.alias("orders")
    .join(products_df.alias("products"), on="product_id", how="left")
    .join(history_summary_df.alias("history"), on="customer_id", how="left")
    .withColumn(
        "order_net_amount",
        F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2),
    )
    .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
    .select(
        "order_id", "customer_id", "order_datetime", "modified_datetime",
        "product_id", "product_name", "product_category", "quantity", "unit_price",
        "discount", "order_net_amount", "order_status", "shipping_country",
        "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
    )
)

# Optional development inspection — uncomment only when needed.
# display(transformed_df)


# 4. Target

Each cloneable target block defines one target DataFrame and the exact named sources that feed it. This keeps Source Drift and Lineage correct when one notebook contains multiple targets.


In [ ]:
WRITE_DATAFRAME = transformed_df
WRITE_SOURCE_NAMES = ("orders", "products", "history")
WRITE_STORE = "unified"
WRITE_SCHEMA = "demo"
WRITE_TABLE = "curated_orders"
WRITE_LOAD_STRATEGY = "overwrite"

# Pick only the source blocks that actually feed this target.
# FabricOps uses this list later for Source Drift and Lineage.
write_sources = [sources[name] for name in WRITE_SOURCE_NAMES]

# Resolve the FabricOps table_id for the target before running target checks.
target_table_id = resolve_table_id(
    store=WRITE_STORE,
    schema=WRITE_SCHEMA,
    table_name=WRITE_TABLE,
)


# 5. Write Preparation / Guardrails

Run Target Schema, Sensitive Data, Source Drift, and Target Data Quality explicitly and in order. Source Drift runs only for the named sources feeding this target. Caller-owned DQ failures and Sensitive Data support mappings stay available for optional inspection or project-owned persistence.


In [ ]:
# Make sure the transformed columns still match the target contract before writing anything.
target_schema_result = check_schema(
    WRITE_DATAFRAME,
    table_id=target_table_id,
    raise_on_failure=True,
)

# Apply the target's sensitive-data rules before DQ and before the final write.
# If tokenization creates a support mapping, keep it available so the project can save it where needed.
sensitive_result = check_sensitive_data(
    WRITE_DATAFRAME,
    table_id=target_table_id,
    raise_on_failure=True,
)
prepared_df = sensitive_result["dataframe"]
support_mapping_df = sensitive_result.get("support_mapping")

# Check whether any source feeding this target has drifted since the last successful target write.
for source in write_sources:
    check_source_drift(
        source["table_id"],
        target_table_id=target_table_id,
        raise_on_failure=True,
    )

# Run the target's DQ rules after sensitive-data handling.
# Keep the checked DataFrame and failed rows available for inspection or project-owned persistence.
target_dq_result = check_dq(
    prepared_df,
    table_id=target_table_id,
    raise_on_failure=True,
)
target_dq_df = target_dq_result.get("dataframe", prepared_df)
target_dq_failed_values = target_dq_result.get("failed_values")

# Optional development inspection — uncomment only when needed.
# display(prepared_df)                     # View the data that is ready to be written
# display(support_mapping_df)              # View token/support mappings when one was created
# display(target_dq_df)                    # View target data with DQ status columns
# display(target_dq_failed_values)         # View rows/values that failed target DQ

# Optional support persistence — save caller-owned support data where your project needs it.
# write_lakehouse_table(
#     support_mapping_df,
#     "sensitive_support_mapping",
#     store="unified",
#     schema="support",
#     mode="append",
# )
# write_warehouse_table(
#     support_mapping_df,
#     schema="support",
#     table_name="sensitive_support_mapping",
#     store="warehouse",
#     mode="append",
# )

# Optional failure persistence — save target DQ failures somewhere your project owns.
# write_lakehouse_table(
#     target_dq_failed_values,
#     "target_dq_failed_values",
#     store="unified",
#     schema="support",
#     mode="append",
# )
# write_warehouse_table(
#     target_dq_failed_values,
#     schema="support",
#     table_name="target_dq_failed_values",
#     store="warehouse",
#     mode="append",
# )


# 6. Write

Publish only after the explicit target checks allow continuation. The governed target load strategy may still be overwrite, append, SCD1, or SCD2 even though this template always reads its sources in full.


In [ ]:
# Write the prepared target and record which source table_ids produced it.
# FabricOps commits Lineage and Source Observation only after this write succeeds.
write_result = pipeline_write(
    prepared_df,
    store=WRITE_STORE,
    schema=WRITE_SCHEMA,
    table_name=WRITE_TABLE,
    load_strategy=WRITE_LOAD_STRATEGY,
    source_table_ids=[source["table_id"] for source in write_sources],
)


# 7. Persisted Target Profile

Profile the complete persisted target after publication; this intentionally reads back state that can differ from an append, partition overwrite, SCD1, or SCD2 input DataFrame.


In [ ]:
# Refresh the profile from the table that was actually written, not just the in-memory DataFrame.
write_profile = profile_table(table_id=write_result["table_id"])

# Optional development inspection — uncomment only when needed.
# display(write_profile["profile"])

# Optional metadata inspection — show what FabricOps knows about the written target.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=write_result["table_id"])
